# OptiCrop Model Training
Train a crop recommendation model using the provided dataset and export the trained artifacts for Flask use.

In [ ]:
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

base_dir = Path.cwd().resolve().parent
data_path = base_dir / 'dataset' / 'Crop_recommendation.csv'
df = pd.read_csv(data_path)
df = df.dropna()
df['label'] = df['label'].astype(str)
X = df[['N','P','K','temperature','humidity','ph','rainfall']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

models = {
    'Logistic Regression': LogisticRegression(max_iter=5000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = {}
for name, model in models.items():
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(name, acc)

best_model_name = max(results, key=results.get)
best_model = models[best_model_name]
best_pipeline = Pipeline([('scaler', StandardScaler()), ('model', best_model)])
best_pipeline.fit(X_train, y_train)

label_encoder = LabelEncoder()
label_encoder.fit(y)

model_dir = base_dir / 'model'
model_dir.mkdir(exist_ok=True)
with (model_dir / 'model.pkl').open('wb') as f:
    pickle.dump(best_pipeline, f)
with (model_dir / 'label_encoder.pkl').open('wb') as f:
    pickle.dump(label_encoder, f)

print('Best model:', best_model_name)